In [3]:
import os
import pandas as pd
import wave
import numpy as np
import matplotlib.pyplot as plt
import librosa
import shutil
import pickle
import pywt
from scipy.stats import kurtosis
import wav_utils
from skimage.filters import threshold_otsu
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Polygon
import pyloudnorm as pyln
import soundfile as sf

In [4]:
dataFolder = "Samples"

n_fft = 4096 

data_files = os.listdir(dataFolder)
data_files.sort()

df_loudness = pd.DataFrame(columns = ['User','Test','Severity','Total_Loudness','Total_RMS','cA5_Loudness',
                                      'D5_Loudness','D4_Loudness','D3_Loudness','D2_Loudness','D1_Loudness',
                                      'cA5_RMS','D5_RMS','D4_RMS','D3_RMS','D2_RMS','D1_RMS'])

for data_file in data_files:

    #if not a wav file, skip
    if ".wav" not in data_file:
        continue

    userName, userTest, userSeverity = wav_utils.getTestInfo(data_file)
    
    # FIXED: Create dict to build single-row DataFrame
    row_data = {
        'User': userName,
        'Test': userTest,
        'Severity': userSeverity
    }

    # path
    wavFilepath = f"{dataFolder}/{data_file}"

    # load data
    raw_audio, sample_rate = sf.read(wavFilepath)

    # Noise reduction method, filter
    filter_lowcut = 80
    filter_highcut = 1800
    filter_order = 8
    filter_btype = "bandpass"  
    filt_audio = wav_utils.filter_denoise(raw_audio, sample_rate, filter_order,
                                    filter_lowcut, filter_highcut, btype=filter_btype)
    
    # Normalise and remove DC component in signal
    x = filt_audio

    # Create a meter
    meter = pyln.Meter(sample_rate)
    loudness = meter.integrated_loudness(x)

    row_data['Total_Loudness'] = loudness
    row_data['Total_RMS'] = np.sqrt(np.median(x**2))
   
    # rebuilding a wav file for each level of DWT
    wavelet='db6'
    coeffs = pywt.wavedec(x, wavelet, level=5)
    wavFiles = []
    for wavLevel in range(len(coeffs)):
        tempCoeffs = coeffs.copy()
        for i in range(len(coeffs)):
            if i != wavLevel:
                tempCoeffs[i] = np.zeros_like(tempCoeffs[i])
        
        wavFiles.append(tempCoeffs)

    wavFiles_recon = []
    for i in range(len(wavFiles)):
        wavFiles_recon.append(pywt.waverec(wavFiles[i], wavelet))
        sf.write(f'{userName}_{userTest}_{userSeverity}_{i}.wav', wavFiles_recon[i], 4000, subtype='PCM_16') 

    # now have a list of reconstructed wav files
    labels = ['cA5','D5','D4','D3','D2','D1']
    for i in range(len(wavFiles_recon)):
        loudness = meter.integrated_loudness(wavFiles_recon[i])
        
        row_data[labels[i] + '_Loudness'] = loudness
        row_data[labels[i] + '_RMS'] = np.sqrt(np.median(wavFiles_recon[i]**2))
    
    # FIXED: Create DataFrame from dict with index=[0]
    df_line = pd.DataFrame(row_data, index=[0])
    df_loudness = pd.concat([df_loudness, df_line], ignore_index=True)

/var/folders/w5/38gvs5991kl2xwv6ry5ys4mw0000gn/T/ipykernel_14891/1429147763.py:78: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_loudness = pd.concat([df_loudness, df_line], ignore_index=True)


In [5]:
df_loudness

,User,Test,Severity,Total_Loudness,Total_RMS,cA5_Loudness,D5_Loudness,D4_Loudness,D3_Loudness,D2_Loudness,D1_Loudness,cA5_RMS,D5_RMS,D4_RMS,D3_RMS,D2_RMS,D1_RMS
0,p000,t01,healthy,-50.918954,0.000254,-inf,-58.486581,-55.272872,-50.572029,-58.361921,-inf,0.000025,0.000183,0.000134,0.000060,0.000020,0.000005
1,p000,t02,healthy,-66.931968,0.000202,-inf,-66.963086,-68.155736,-inf,-inf,-inf,0.000022,0.000153,0.000087,0.000040,0.000024,0.000006
2,p001,t01,mild,-54.941500,0.000274,-inf,-64.640102,-57.751077,-57.988986,-66.291548,-inf,0.000023,0.000191,0.000138,0.000059,0.000015,0.000004
3,p002,t01,severe,-61.457649,0.000329,-inf,-66.275720,-63.121967,-59.374622,-67.159611,-inf,0.000031,0.000236,0.000165,0.000071,0.000016,0.000004
4,p002,t02,severe,-54.941500,0.000274,-inf,-64.640102,-57.751077,-57.988986,-66.291548,-inf,0.000023,0.000191,0.000138,0.000059,0.000015,0.000004
5,p002,t03,severe,-53.425964,0.000329,-inf,-56.486188,-57.246886,-59.848385,-68.149590,-inf,0.000037,0.000255,0.000153,0.000058,0.000014,0.000004
6,p003,t01,moderate,-56.581606,0.000432,-inf,-62.552513,-61.477507,-59.531966,-64.269018,-inf,0.000034,0.000270,0.000196,0.000161,0.000057,0.000007
7,p003,t02,moderate,-52.698280,0.000317,-66.352966,-52.143342,-56.346692,-56.209876,-62.414783,-inf,0.000026,0.000198,0.000168,0.000099,0.000035,0.000005
8,p004,t01,moderate,-55.836413,0.000295,-inf,-63.691016,-56.293464,-58.940953,-65.101007,-inf,0.000025,0.000200,0.000150,0.000064,0.000017,0.000005
9,p004,t02,moderate,-48.095752,0.000407,-66.597791,-53.818778,-49.499823,-54.532365,-64.252883,-inf,0.000039,0.000292,0.000203,0.000071,0.000018,0.000005
